# Análisis Exploratorio

Lectura compras.csv

In [0]:
# Read the CSV file, skipping the header
with open('/Volumes/workspace/default/prueba_bavaria/compras.csv', 'r') as f:
    header = f.readline()
    rows = f.readlines()

# Replace | with , in the rows
rows = [row.replace('|', ',') for row in rows]

# Write the header and modified rows to a new CSV file
output_path = '/Volumes/workspace/default/prueba_bavaria/compras_clean.csv'
with open(output_path, 'w') as f:
    f.write(header)
    f.writelines(rows)

# Read the cleaned CSV into a Spark DataFrame
df_compras = spark.read.format("csv").option("header", "true").load(output_path)

# Create Databricks SQL table
df_compras.write.format("delta").mode("overwrite").saveAsTable("workspace.default.prueba_bavaria_compras")

In [0]:
display(df_compras)
df_compras.printSchema()

Lectura productos.csv

In [0]:
# Read the CSV file, skipping the header
with open('/Volumes/workspace/default/prueba_bavaria/productos.csv', 'r') as f:
    header = f.readline()
    rows = f.readlines()

# Replace | with , in the rows
rows = [row.replace('|', ',') for row in rows]

# Write the header and modified rows to a new CSV file
output_path = '/Volumes/workspace/default/prueba_bavaria/productos_clean.csv'
with open(output_path, 'w') as f:
    f.write(header)
    f.writelines(rows)

# Read the cleaned CSV into a Spark DataFrame
df_productos = spark.read.format("csv").option("header", "true").load(output_path)

# Create Databricks SQL table
df_productos.write.format("delta").mode("overwrite").saveAsTable("workspace.default.prueba_bavaria_productos")

In [0]:
display(df_productos)
df_productos.printSchema()

Join Tablas

In [0]:
df_compras = spark.table("workspace.default.prueba_bavaria_compras")
df_productos = spark.table("workspace.default.prueba_bavaria_productos")

df = df_compras.join(df_productos, on="SKU id", how="left")

In [0]:
from pyspark.sql.functions import col, to_date

df = df.withColumn("GMV", col("GMV").cast("double")) \
       .withColumn("Unit_Quantity", col("Unit_Quantity").cast("int")) \
       .withColumn("Volume", col("Volume").cast("double")) \
       .withColumn("SKU SIZE", col("SKU SIZE").cast("double")) \
       .withColumn("Order_Date", to_date("Order_Date"))

In [0]:
df.printSchema()
df.describe().show()

In [0]:
%sql
SELECT * FROM workspace.default.prueba_bavaria_compras

Databricks visualization. Run in Databricks to view.

In [0]:
df_pandas = df.toPandas()
display(df_pandas)
   